# SentinelVision AI
## Notebook 04 — Baseline Anomaly Models

### Objective

Notebook 03 created reusable clip embeddings from real UCF-Crime video clips.

This notebook trains the first anomaly detection models using those embeddings.

We will build:

1. a simple distance-threshold baseline,
2. an Isolation Forest model.

The goal is to establish a clean baseline before moving to deep learning.

The flow is:

`clip embeddings → train on normal training clips → anomaly scores → validation threshold → test evaluation`

In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

#### Load embeddings

In [2]:
embeddings_path = Path("../data/embeddings/clip_embeddings_sample.parquet")

embeddings = pd.read_parquet(embeddings_path)

embeddings.shape

(498, 23)

In [3]:
embeddings.head()

,clip_id,video_id,category,binary_label,split,start_seconds,end_seconds,feature_000,feature_001,feature_002,...,feature_006,feature_007,feature_008,feature_009,feature_010,feature_011,feature_012,feature_013,feature_014,feature_015
0,Shooting022_x264_clip_00010,Shooting022_x264,Shooting,1,test,40.0,44.0,0.416163,0.386835,0.374141,...,0.394131,0.225522,0.000269,0.000158,0.000236,0.000118,0.000140,0.000204,0.000162,0.000087
1,Vandalism041_x264_clip_00021,Vandalism041_x264,Vandalism,1,test,84.0,88.0,0.257803,0.171437,0.062419,...,0.184771,0.186079,0.002649,0.001979,0.000456,0.003085,0.002011,0.000281,0.001935,0.001966
2,Robbery130_x264_clip_00002,Robbery130_x264,Robbery,1,test,8.0,12.0,0.425955,0.425692,0.420834,...,0.425198,0.173660,0.003575,0.002963,0.002644,0.006365,0.006437,0.005010,0.003006,0.006305
3,Normal_Videos_641_x264_clip_00025,Normal_Videos_641_x264,Normal,0,test,100.0,104.0,0.289380,0.271774,0.224172,...,0.271585,0.246034,0.007098,0.005536,0.003933,0.000932,0.002951,0.001615,0.005603,0.002558
4,Abuse015_x264_clip_00012,Abuse015_x264,Abuse,1,test,48.0,52.0,0.503250,0.506105,0.498351,...,0.504288,0.403972,0.000653,0.000701,0.000608,0.000398,0.000426,0.000355,0.000664,0.000405


#### Identify feature columns

In [4]:
feature_columns = [
    column
    for column in embeddings.columns
    if column.startswith("feature_")
]

len(feature_columns), feature_columns[:5]

(16,
 ['feature_000', 'feature_001', 'feature_002', 'feature_003', 'feature_004'])

In [5]:
metadata_columns = [
    column
    for column in embeddings.columns
    if column not in feature_columns
]

metadata_columns

['clip_id',
 'video_id',
 'category',
 'binary_label',
 'split',
 'start_seconds',
 'end_seconds']

#### Split embeddings

In [6]:
train_data = embeddings[
    embeddings["split"] == "train"
].copy()

validation_data = embeddings[
    embeddings["split"] == "validation"
].copy()

test_data = embeddings[
    embeddings["split"] == "test"
].copy()

print("Train:", train_data.shape)
print("Validation:", validation_data.shape)
print("Test:", test_data.shape)

Train: (166, 23)
Validation: (166, 23)
Test: (166, 23)


In [7]:
train_data["binary_label"].value_counts()

binary_label
1    162
0      4
Name: count, dtype: int64

In [8]:
validation_data["binary_label"].value_counts()

binary_label
1    158
0      8
Name: count, dtype: int64

In [10]:
test_data["binary_label"].value_counts()

binary_label
1    165
0      1
Name: count, dtype: int64

#### Train only on normal training clips

In [11]:
normal_train_data = train_data[
    train_data["binary_label"] == 0
].copy()

normal_train_data.shape

(4, 23)

In [12]:
X_train_normal = normal_train_data[feature_columns].values
X_validation = validation_data[feature_columns].values
X_test = test_data[feature_columns].values

y_validation = validation_data["binary_label"].values
y_test = test_data["binary_label"].values

## Part A — Distance Threshold Baseline

#### Scale features

In [13]:
scaler = StandardScaler()

X_train_normal_scaled = scaler.fit_transform(
    X_train_normal
)

X_validation_scaled = scaler.transform(
    X_validation
)

X_test_scaled = scaler.transform(
    X_test
)

#### Compute normal centroid

In [14]:
normal_centroid = X_train_normal_scaled.mean(
    axis=0
)

normal_centroid.shape

(16,)

In [15]:
def centroid_distance_scores(
    X: np.ndarray,
    centroid: np.ndarray,
) -> np.ndarray:
    """
    Compute distance from each sample to the normal centroid.

    Larger distance = more anomalous.
    """
    distances = np.linalg.norm(
        X - centroid,
        axis=1,
    )

    return distances

In [16]:
validation_distance_scores = centroid_distance_scores(
    X_validation_scaled,
    normal_centroid,
)

test_distance_scores = centroid_distance_scores(
    X_test_scaled,
    normal_centroid,
)

#### Pick threshold using validation set

In [17]:
precision, recall, thresholds = precision_recall_curve(
    y_validation,
    validation_distance_scores,
)

f1_scores = []

for threshold in thresholds:
    predictions = (
        validation_distance_scores >= threshold
    ).astype(int)

    score = f1_score(
        y_validation,
        predictions,
        zero_division=0,
    )

    f1_scores.append(score)

best_index = int(np.argmax(f1_scores))
best_distance_threshold = thresholds[best_index]
best_validation_f1 = f1_scores[best_index]

best_distance_threshold, best_validation_f1

(np.float64(1.9641133284166767), 0.9753086419753086)

#### Evaluate distance baseline on test set

In [18]:
test_distance_predictions = (
    test_distance_scores >= best_distance_threshold
).astype(int)

In [19]:
distance_results = {
    "model": "Distance Threshold Baseline",
    "roc_auc": roc_auc_score(
        y_test,
        test_distance_scores,
    ),
    "pr_auc": average_precision_score(
        y_test,
        test_distance_scores,
    ),
    "f1": f1_score(
        y_test,
        test_distance_predictions,
        zero_division=0,
    ),
}

distance_results

{'model': 'Distance Threshold Baseline',
 'roc_auc': 0.509090909090909,
 'pr_auc': 0.9959607591706074,
 'f1': 0.9847094801223242}

In [20]:
print(
    classification_report(
        y_test,
        test_distance_predictions,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.99      0.98      0.98       165

    accuracy                           0.97       166
   macro avg       0.50      0.49      0.49       166
weighted avg       0.99      0.97      0.98       166



## Part B — Isolation Forest

#### Train Isolation Forest

In [21]:
isolation_forest = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

isolation_forest.fit(
    X_train_normal_scaled
)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


#### Score validation and test data

In [22]:
# sklearn returns higher decision_function values for more normal samples.
# We multiply by -1 so higher score means more anomalous.
validation_iforest_scores = -isolation_forest.decision_function(
    X_validation_scaled
)

test_iforest_scores = -isolation_forest.decision_function(
    X_test_scaled
)

#### Pick Isolation Forest threshold using validation set

In [23]:
precision, recall, thresholds = precision_recall_curve(
    y_validation,
    validation_iforest_scores,
)

f1_scores = []

for threshold in thresholds:
    predictions = (
        validation_iforest_scores >= threshold
    ).astype(int)

    score = f1_score(
        y_validation,
        predictions,
        zero_division=0,
    )

    f1_scores.append(score)

best_index = int(np.argmax(f1_scores))
best_iforest_threshold = thresholds[best_index]
best_iforest_validation_f1 = f1_scores[best_index]

best_iforest_threshold, best_iforest_validation_f1

(np.float64(-0.09430644627622997), 0.9753086419753086)